<a href="https://colab.research.google.com/github/quiquefluque/Python-Finace/blob/main/Largo_plazo_Msci_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#Esto no es un bot (esto vale para ver si invierto en un momento dado cuanta cantidad tendre (intervalo) en un momento del futuro con una probablidad)
#1.Importar
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#2.Crear tabla
df = yf.download('URTH', period='5y')
df = df[['Close']].copy()
df.columns = ['Precio']

#3.análisis pasado
df["Retornos"]=df["Precio"].pct_change()
media_retornos=df["Retornos"].mean()
vol_diaria = df["Retornos"].std()

########## hacernos una idea anual
media_anual = media_retornos * 252
vol_anual = vol_diaria * np.sqrt(252)
print(f"--- ADN MSCI WORLD (AÑOS) ---")
print(f"Retorno medio anual: {media_anual*100:.2f}%")
print(f"Volatilidad anual: {vol_anual*100:.2f}%")


# 4.Monte carlo (futuros hipotéticos)
diasbursatiles= 252*5#(el 5 son los años a futuro)
numfuturos=1000
precio_actual=df['Precio'].iloc[-1]
df_futura={}
for i in range (numfuturos):
  retornosfuturos=np.random.normal(media_retornos,vol_diaria,size=diasbursatiles)
  precios_simulados = precio_actual * (1 + retornosfuturos).cumprod(axis=0)
  df_futura[f"Simulacion número {i}"] = precios_simulados #añado columna por cada simulacion con los precios

#  5.Cartera
inversion_inicial = 1000  # <--- Pon aquí lo que has invertido
df_simulaciones = pd.DataFrame(df_futura)

# Calculamos el multiplicador: Precio Final / Precio Inicial
# Esto nos dice cuánto ha crecido (o caído) cada euro
multiplicadores_finales = df_simulaciones.iloc[-1] / precio_actual #da lista

# Calculamos el dinero final en cada mundo
dinero_final = inversion_inicial * multiplicadores_finales #da lista

# --- INTERVALOS DE CONFIANZA ---
# np.percentile ordena los 1000 resultados de menos a mas y da el dato en una posición(95=95 porciento de 1000, solo el 5 porciento restante se gana mas de eso)

suelo_95 = np.percentile(dinero_final, 5)   # El 95% de las veces ganarás más que esto
techo_5 = np.percentile(dinero_final, 95)    # Solo el 5% de las veces ganarás más que esto
mediana = np.percentile(dinero_final, 50)   # El resultado más central


print(f"\n--- PROYECCIÓN DE TU INVERSIÓN ({inversion_inicial}€) ---")
print(f"Mediana (Lo más probable): {mediana:.2f}€")
print(f"El 95% de las veces ganare mas de: {suelo_95:.2f}€")
print(f"Solo el 5% de las veces ganare más de: {techo_5:.2f}€")

# ¿Qué probabilidad hay de perder dinero?
prob_perdida = (dinero_final < inversion_inicial).mean() * 100
print(f"Probabilidad de estar en pérdidas tras 1 año: {prob_perdida:.2f}%")

#Inversor activo vs inversor que no hace nada
lista_final_nada = []
lista_final_activo = []
ventana_de_seguridad= 200
for col in df_simulaciones.columns:
  #inversor que no hace nada
  precios_mundo=df_simulaciones[col]
  final_p = (precios_mundo.iloc[-1] / precio_actual) * inversion_inicial
  lista_final_nada.append(final_p)
  #inversor activo
  media_m = precios_mundo.rolling(window=ventana_de_seguridad).mean()
  señal = (precios_mundo > media_m).astype(int).shift(1).fillna(0)#crea señal de 1 si precio>media =1,1 compra con el dinero que hay, 0 vende, el shift coje la columna de señales y la manda para abajo, se compra o vende el dia siguiente
  ret_diarios = precios_mundo.pct_change().fillna(0)
  ret_robot = ret_diarios * señal
  capital_robot = inversion_inicial * (1 + ret_robot).cumprod()
  lista_final_activo.append(capital_robot.iloc[-1])

final_piedra = np.array(lista_final_nada)
final_robot = np.array(lista_final_activo)

print(f"\n--- COMPARATIVA A 5 AÑOS ({inversion_inicial}€) ---")
print(f"PIEDRA | Mediana: {np.percentile(final_piedra, 50):.2f}€ | Suelo 95%: {np.percentile(final_piedra, 5):.2f}€")
print(f"ROBOT  | Mediana: {np.percentile(final_robot, 50):.2f}€ | Suelo 95%: {np.percentile(final_robot, 5):.2f}€")

# ¿Quién es más eficiente?
prob_ganar_a_piedra = (final_robot > final_piedra).mean() * 100
print(f"\nProbabilidad de que el Robot mejore al Buy&Hold: {prob_ganar_a_piedra:.2f}%")



/tmp/ipykernel_194/1171045365.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('URTH', period='5y')
[*********************100%***********************]  1 of 1 completed


--- ADN MSCI WORLD (AÑOS) ---
Retorno medio anual: 11.46%
Volatilidad anual: 16.05%

--- PROYECCIÓN DE TU INVERSIÓN (1000€) ---
Mediana (Lo más probable): 1660.33€
El 95% de las veces ganare mas de: 865.47€
Solo el 5% de las veces ganare más de: 2968.53€
Probabilidad de estar en pérdidas tras 1 año: 8.70%

--- COMPARATIVA A 5 AÑOS (1000€) ---
PIEDRA | Mediana: 1660.33€ | Suelo 95%: 865.47€
ROBOT  | Mediana: 1310.55€ | Suelo 95%: 821.82€

Probabilidad de que el Robot mejore al Buy&Hold: 14.00%
